# Regresi skor / engagement / ordinal

Baca metrik soal: RMSE, MAE, RMSLE dan QWK membutuhkan keputusan berbeda. Jangan menganggap semua target float adalah regresi.

Kode lengkap dari pipeline dan snippet asli. Ikuti urutan tahap dan ubah konfigurasi awal sesuai data. Alternatif tersedia pada cheatsheet utama.

## Setup & konfigurasi

Satu-satunya sel yang wajib diubah. Jalankan paling awal.

In [ ]:
import os
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")
from pathlib import Path
import importlib.metadata as md

def env_report(models_dir=None):
    import torch
    for p in ("torch", "transformers", "scikit-learn", "pandas", "numpy", "sentence-transformers", "lightgbm"):
        try:
            print(p, md.version(p))
        except md.PackageNotFoundError:
            print(p, "TIDAK ADA")
    gpu = torch.cuda.is_available()
    print("cuda", gpu, torch.cuda.get_device_name(0) if gpu else "")
    if gpu:
        print("VRAM GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "| bf16", torch.cuda.is_bf16_supported())
    if models_dir and Path(models_dir).exists():
        print("model lokal:", sorted(p.name for p in Path(models_dir).iterdir() if p.is_dir()))
    return gpu

import os
import random
import numpy as np
import torch

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

# ===== jalankan =====
import re, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
MODELS = Path(os.environ.get("COMP_ROOT", r"C:\gemastik")) / "models"
SEED = 42
seed_everything(SEED)
env_report(MODELS)

TRAIN, TEST, SAMPLE = "train.csv", "test.csv", "sample_submission.csv"
TEXT, TARGET = "text", "label"
PREPROCESS = "tfidf"
EXPERIMENTS = [("tfidf_word", "ridge"), ("lsa", "lgbm_reg")]   # target berekor: ridge_log. MAE: lgbm_l1
ORDINAL = True                                   # True bila skor diskret dan metrik QWK
USE_BERT = False
BERT_MODEL, MAX_LEN, BATCH = MODELS / "indobertweet", 128, 32


## Muat data

Baca file, otomatis mencoba encoding, simpan parquet.

In [ ]:
from pathlib import Path
import pandas as pd

def read_any(path, cache=True, **kw):
    """df = read_any('train.csv'). Lain kali otomatis membaca train.parquet."""
    p = Path(path)
    if p.suffix == ".parquet":
        return pd.read_parquet(p, **kw)
    if cache and p.with_suffix(".parquet").exists():
        return pd.read_parquet(p.with_suffix(".parquet"))
    if p.suffix in (".jsonl", ".json"):
        return pd.read_json(p, lines=p.suffix == ".jsonl", **kw)
    if p.suffix in (".xlsx", ".xls"):
        return pd.read_excel(p, **kw)
    sep = kw.pop("sep", "\t" if p.suffix == ".tsv" else ",")
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(p, sep=sep, encoding=enc, on_bad_lines="warn", **kw)
            break
        except UnicodeDecodeError:
            continue
    if cache:
        df.to_parquet(p.with_suffix(".parquet"), index=False)
    return df

# ===== jalankan =====
train, test = read_any(TRAIN), read_any(TEST)
print("train", train.shape, "| test", test.shape)
print(train.head(3).to_string())


## EDA

Skew dan target diskret.

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_predict

def _nk(s):
    return s.fillna("").astype(str).str.lower().str.replace(r"\W+", " ", regex=True).str.strip()

def eda_report(train, test=None, text="text", target=None, time=None, n_show=8, sample=20000, seed=42):
    """Kembalikan dict ringkasan berisi saran dan rekomendasi max_words. target: nama kolom atau daftar kolom."""
    tips, out = [], {}
    print("=== PROFIL ===\ntrain", train.shape, "| test", None if test is None else test.shape)
    print(pd.DataFrame({"dtype": train.dtypes.astype(str), "null": train.isna().mean().round(4), "nunique": train.nunique()}).to_string())
    if test is not None:
        print("kolom hanya di train:", sorted(set(train.columns) - set(test.columns)))
    tx = train[text].fillna("").astype(str)
    print("\n=== CONTOH ===")
    for t in train[text].sample(min(n_show, len(train)), random_state=seed):
        print(" -", str(t)[:180])
    key = _nk(train[text])
    out["kosong"], out["duplikat"] = round(float(key.eq("").mean()), 4), round(float(key.duplicated().mean()), 4)
    print("\n=== KUALITAS ===\nkosong", out["kosong"], "| duplikat ternormalisasi", out["duplikat"], "| punya URL", round(tx.str.contains(r"https?://").mean(), 3),
          "| mention", round(tx.str.contains(r"@\w").mean(), 3), "| tanda baca akhir kalimat", round(tx.str.contains(r"[.!?]").mean(), 3))
    print("teks paling sering:", key.value_counts().head(3).to_dict())
    if out["duplikat"] > .01:
        tips.append("Duplikat > 1%: pakai group fold dan laporkan skor random serta grup.")
    w = tx.str.split().str.len()
    q = w.quantile([.5, .9, .95, .99]).round(0)
    out["max_words"] = int(q[.95])
    print("\n=== PANJANG (kata) ===\n", q.to_dict(), "| char p50", int(tx.str.len().median()))
    if q[.5] < 5:
        tips.append("Teks sangat pendek: utamakan TF-IDF karakter.")
    if q[.95] > 256:
        tips.append("Teks panjang: pertimbangkan head+tail dan chunk.")
    if target is not None:
        print("\n=== TARGET ===")
        cols = target if isinstance(target, (list, tuple)) else [target]
        if len(cols) > 1:
            Y = train[cols].to_numpy()
            print(pd.DataFrame({"pct": train[cols].mean().round(4)}).T.to_string(), "\nlabel per baris", round(float(Y.sum(1).mean()), 3))
        else:
            s = train[cols[0]]
            if pd.api.types.is_numeric_dtype(s) and s.nunique() > 20:
                print(s.describe().round(3).to_dict(), "skew", round(s.skew(), 3))
                if s.skew() > 1 and s.min() >= 0:
                    tips.append("Target berekor panjang: latih pada log1p.")
            else:
                vc = s.value_counts(dropna=False)
                print((vc / len(s)).round(4).to_dict())
                out["rasio_imbalance"] = round(float(vc.max() / max(vc.min(), 1)), 1)
                if out["rasio_imbalance"] > 20:
                    tips.append("Imbalance parah: metrik macro-F1/PR-AUC, decision layer dari OOF.")
                if s.nunique() <= 50 and s.notna().all():
                    v = TfidfVectorizer(max_features=30000, min_df=3, sublinear_tf=True)
                    X = v.fit_transform(tx.iloc[:sample])
                    m = LogisticRegression(C=4, max_iter=1000).fit(X, s.iloc[:sample].astype(str))
                    names, coef = v.get_feature_names_out(), m.coef_ if m.coef_.shape[0] > 1 else np.vstack([-m.coef_[0], m.coef_[0]])
                    print("kata khas (cek artefak):")
                    for j, c in enumerate(m.classes_):
                        print(f"  {c}:", [names[i] for i in coef[j].argsort()[::-1][:10]])
    cv = CountVectorizer(token_pattern=r"(?u)\b\w+\b", binary=True)
    X = cv.fit_transform(tx.sample(min(sample, len(tx)), random_state=seed))
    df_ = np.asarray(X.sum(0)).ravel()
    print("\n=== KOSAKATA ===\nvocab", X.shape[1], "| kata muncul sekali", round(float((df_ == 1).mean()), 3))
    if test is not None and text in test:
        te = test[text].fillna("").astype(str)
        vt = CountVectorizer(token_pattern=r"(?u)\b\w+\b").fit(te.sample(min(sample, len(te)), random_state=seed))
        cnt = np.asarray(vt.transform(te.sample(min(sample, len(te)), random_state=seed)).sum(0)).ravel()
        oov = np.array([x not in cv.vocabulary_ for x in vt.get_feature_names_out()])
        out["oov"] = round(float(cnt[oov].sum() / max(cnt.sum(), 1)), 3)
        tk = set(key)
        out["overlap"] = round(float(_nk(te).isin(tk).mean()), 4)
        print("OOV token test", out["oov"], "| teks test ada di train", out["overlap"])
        if out["oov"] > .2:
            tips.append("OOV tinggi: TF-IDF karakter atau FastText.")
        n = min(5000, len(tx), len(te))
        t2 = list(tx.sample(n, random_state=seed)) + list(te.sample(n, random_state=seed))
        lab = np.r_[np.zeros(n), np.ones(n)]
        Xa = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000).fit_transform(t2)
        out["adversarial_auc"] = round(float(roc_auc_score(lab, cross_val_predict(LogisticRegression(max_iter=500), Xa, lab, cv=3, method="predict_proba")[:, 1])), 4)
        print("AUC adversarial", out["adversarial_auc"], "| panjang p50 test", int(te.str.split().str.len().median()))
        if out["adversarial_auc"] > .7:
            tips.append("Train dan test berbeda: cek kata pembeda, pertimbangkan split waktu/sumber.")
    if time and time in train:
        t = pd.to_datetime(train[time], errors="coerce")
        print("\n=== WAKTU ===\nrentang", t.min(), "s.d.", t.max(), "| gagal parse", round(float(t.isna().mean()), 3))
        if test is not None and time in test:
            tt = pd.to_datetime(test[time], errors="coerce")
            print("test", tt.min(), "s.d.", tt.max())
            if tt.min() > t.median():
                tips.append("Test lebih baru dari sebagian besar train: pakai time split.")
    out["saran"] = tips
    print("\n=== SARAN ===", *(tips or ["Tidak ada peringatan khusus."]), sep="\n - ")
    return out

import numpy as np
import pandas as pd

def label_table(y):
    s = pd.Series(np.asarray(y)).value_counts()
    print("rasio mayor:minor", round(s.max() / s.min(), 1))
    return pd.DataFrame({"n": s, "pct": (s / s.sum()).round(4)})

def multilabel_table(Y, names=None):
    Y = np.asarray(Y)
    names = names if names is not None else [f"l{i}" for i in range(Y.shape[1])]
    print("rata-rata label per baris", round(Y.sum(1).mean(), 3), "| baris tanpa label", round((Y.sum(1) == 0).mean(), 3))
    co = pd.DataFrame(Y.T @ Y, index=names, columns=names)
    return pd.DataFrame({"n": Y.sum(0), "pct": Y.mean(0).round(4)}, index=names), co

def target_table(y):
    s = pd.Series(np.asarray(y, dtype=float))
    print("skew", round(s.skew(), 3), "| nol", round((s == 0).mean(), 3), "| unik", s.nunique(), "| disarankan log1p" if s.skew() > 1 and s.min() >= 0 else "")
    return s.describe(percentiles=[.01, .5, .99]).round(3)

# ===== jalankan =====
report = eda_report(train, test, TEXT, TARGET)
print(target_table(train[TARGET]))


## Cleaning data

Dedup: nilai target dirata-rata tidak, baris pertama dipertahankan.

In [ ]:
import re
import numpy as np
import pandas as pd

_PSEUDO = {"", "nan", "none", "null", "-", "--", "n/a", "na", "[sticker]"}

def _groups_minhash(keys, threshold=.8, num_perm=64, shingle=5):
    from datasketch import MinHash, MinHashLSH
    lsh, par = MinHashLSH(threshold=threshold, num_perm=num_perm), list(range(len(keys)))
    def find(x):
        while par[x] != x:
            par[x] = par[par[x]]; x = par[x]
        return x
    for i, t in enumerate(keys):
        m = MinHash(num_perm=num_perm)
        for j in range(max(len(t) - shingle + 1, 1)):
            m.update(t[j:j + shingle].encode())
        for k in lsh.query(m):
            par[find(i)] = find(int(k))
        lsh.insert(str(i), m)
    return pd.factorize(np.array([find(i) for i in range(len(keys))]))[0]

def clean_dataset(df, text="text", target=None, dedup=True, near_dup=False, conflict=None, drop_empty=True, near_threshold=.8):
    """Kembalikan df bersih dengan kolom _group untuk fold. conflict: 'majority', 'drop', atau None."""
    n0 = len(df)
    df = df.copy()
    df[text] = df[text].fillna("").astype(str).str.strip()
    if drop_empty:
        df = df[~df[text].str.lower().isin(_PSEUDO)]
    key = df[text].str.lower().str.replace(r"\W+", " ", regex=True).str.strip()
    df["_group"] = pd.factorize(key)[0]
    if near_dup:
        df["_group"] = _groups_minhash(key.tolist(), near_threshold)
    n_conf = 0
    if target is not None and conflict:
        g = df.groupby(key)[target]
        bad = g.transform("nunique") > 1
        n_conf = int(bad.sum())
        if conflict == "majority":
            df[target] = g.transform(lambda s: s.mode().iloc[0])
        elif conflict == "drop":
            df, key = df[~bad], key[~bad]
    if dedup:
        # Konflik label tetap utuh dan satu grup; tidak memilih label pertama diam-diam.
        keep = ~pd.DataFrame({'key':key,'target':df[target]}).duplicated() if target is not None and conflict is None else ~key.duplicated()
        df = df[keep]
    print(f"baris {n0} -> {len(df)} | konflik label {n_conf} | grup {df._group.nunique()}")
    return df.reset_index(drop=True)

def clean_test(df, text="text"):
    """Test tidak dibuang: hanya isi kosong."""
    df = df.copy()
    df[text] = df[text].fillna("").astype(str)
    return df

# ===== jalankan =====
train, test = clean_dataset(train, TEXT, conflict=None), clean_test(test, TEXT)


## Preprocessing

Ganti preset di Setup.

In [ ]:
import html
import re
import unicodedata

SLANG = {"yg": "yang", "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "tdk": "tidak", "bgt": "banget", "tp": "tapi", "klo": "kalau", "kalo": "kalau",
         "udh": "sudah", "dah": "sudah", "dpt": "dapat", "jg": "juga", "jd": "jadi", "bnyk": "banyak", "karna": "karena", "krn": "karena", "utk": "untuk", "jgn": "jangan",
         "org": "orang", "dr": "dari", "lgi": "lagi", "lg": "lagi", "bnr": "benar", "sdh": "sudah", "dgn": "dengan", "aja": "saja"}
STOPWORDS = set("""ada adalah agar akan aku anda antara apa apakah atau bagi bahwa banyak beberapa begitu bila bisa dalam dan dari dengan di dia dua hal hanya harus ia ini itu
jadi jika juga kami kamu karena ke kepada ketika kita lagi lain lebih maka masih mereka mungkin nya oleh pada para saat saja sama sampai sangat satu saya sebagai sebuah sedang
sehingga sekali selalu semua seperti setelah sudah supaya tapi telah tentang tersebut tetapi untuk yaitu yang""".split())   # sengaja tanpa tidak, bukan, jangan, belum, kurang
NEGATION = "tidak|tak|bukan|jangan|belum|kurang|tanpa"
_MOJIBAKE = re.compile("[\u00c3\u00c2\u00e2\u00f0]")

PRESETS = {
    # keep = biarkan, token = ganti penanda, remove = hapus
    "none":        dict(unicode=False, html=False, url="keep", email="keep", phone="keep", mention="keep", hashtag="keep", emoji="keep", number="keep", lower=False, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
    "tfidf":       dict(unicode=True, html=True, url="token", email="token", phone="token", mention="token", hashtag="keep", emoji="keep", number="keep", lower=True, repeat=True, punct="keep", slang=False, negation=False, stopword=False),
    "social":      dict(unicode=True, html=True, url="token", email="token", phone="token", mention="token", hashtag="word", emoji="text", number="keep", lower=True, repeat=True, punct="keep", slang=True, negation=False, stopword=False),
    "topic":       dict(unicode=True, html=True, url="remove", email="remove", phone="remove", mention="remove", hashtag="word", emoji="remove", number="remove", lower=True, repeat=True, punct="remove", slang=True, negation=False, stopword=True),
    "transformer": dict(unicode=True, html=True, url="keep", email="keep", phone="keep", mention="keep", hashtag="keep", emoji="keep", number="keep", lower=False, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
    "indobertweet": dict(unicode=True, html=True, url="HTTPURL", email="keep", phone="keep", mention="@USER", hashtag="keep", emoji="text", number="keep", lower=True, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
}

def _sub(text, pattern, how, token):
    return text if how == "keep" else re.sub(pattern, " " if how == "remove" else f" {token if how == 'token' else how} ", text)

def preprocess(text, preset="tfidf", slang_dict=SLANG, stopwords=STOPWORDS, **override):
    """preprocess('Halo http://a.id', 'social'). Ubah satu langkah: preprocess(t, 'tfidf', emoji='remove')."""
    c = {**PRESETS[preset], **override}
    t = "" if text is None or text != text else str(text)
    if c["unicode"]:
        if _MOJIBAKE.search(t):                         # teks rusak encoding
            import ftfy
            t = ftfy.fix_text(t)
        t = unicodedata.normalize("NFKC", t).replace("​", "")
    if c["html"]:
        t = re.sub(r"<[^>]+>", " ", html.unescape(t))
    t = _sub(t, r"[\w.+-]+@[\w-]+\.[\w.-]+", c["email"], "EMAIL")          # email sebelum mention
    t = _sub(t, r"https?://\S+|www\.\S+", c["url"], "URL")
    t = _sub(t, r"(?<!\d)(?:\+?62|0)8\d{7,11}(?!\d)", c["phone"], "TELEPON")
    t = _sub(t, r"@\w+", c["mention"], "USER")
    if c["hashtag"] == "word":
        t = re.sub(r"#(\w+)", r" \1 ", t)
    elif c["hashtag"] != "keep":
        t = _sub(t, r"#\w+", c["hashtag"], "HASHTAG")
    if c["emoji"] != "keep":
        import emoji
        if c["emoji"] == "id":                          # emoji jadi kata Indonesia
            from indoNLP.preprocessing import emoji_to_words
            t = emoji_to_words(t, delimiter=(" ", " "))
        else:
            t = emoji.demojize(t, delimiters=(" ", " ")) if c["emoji"] == "text" else emoji.replace_emoji(t, " " if c["emoji"] == "remove" else " EMOJI ")
    t = _sub(t, r"\d+(?:[.,]\d+)*", c["number"], "ANGKA")
    if c["lower"]:
        keep = {m: m for m in ("HTTPURL", "@USER", "URL", "EMAIL", "TELEPON", "USER", "ANGKA", "EMOJI", "HASHTAG") if m in t}
        t = t.lower()
        for m in keep:
            t = re.sub(r"(?<!\w)" + re.escape(m.lower()) + r"(?!\w)", m, t)
    if c["repeat"]:
        t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    if c["punct"] == "remove":
        t = re.sub(r"[^\w\s]", " ", t)
    if c["slang"]:
        t = re.sub(r"\b\w+\b", lambda m: slang_dict.get(m.group(0).lower(), m.group(0)), t)
    if c["negation"]:
        t = re.sub(rf"\b({NEGATION})\s+(\w+)", r"\1_\2", t, flags=re.I)
    if c["stopword"]:
        t = " ".join(w for w in t.split() if w.lower() not in stopwords)
    return re.sub(r"\s+", " ", t).strip()

def preprocess_corpus(texts, preset="tfidf", stem=False, slang_csv=None, **override):
    """Versi cepat: teks identik hanya diproses sekali, stem per kata unik. Kembalikan list."""
    slang = dict(SLANG)
    if slang_csv:
        import pandas as pd
        lex = pd.read_csv(slang_csv)
        slang.update(dict(zip(lex["slang"].astype(str), lex["formal"].astype(str))))
    texts = list(texts)
    cache = {t: preprocess(t, preset, slang_dict=slang, **override) for t in {x if x == x else None for x in texts}}
    out = [cache[t if t == t else None] for t in texts]
    if stem:
        from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
        st = StemmerFactory().create_stemmer()
        m = {w: st.stem(w) for w in {w for d in out for w in d.split()}}
        out = [" ".join(m[w] for w in d.split()) for d in out]
    return out

def text_versions(df, text="text", slang_csv=None):
    """Tiga versi agar efek normalisasi bisa diuji: text_raw, text_clean (makna utuh), text_norm (slang dinormalkan)."""
    d = df.copy()
    d["text_raw"] = d[text]
    d["text_clean"] = preprocess_corpus(d[text], "tfidf")
    d["text_norm"] = preprocess_corpus(d[text], "social", slang_csv=slang_csv)
    return d

# ===== jalankan =====
texts, test_texts = preprocess_corpus(train[TEXT], PREPROCESS), preprocess_corpus(test[TEXT], PREPROCESS)


## Fitur

Nama vectorizer di EXPERIMENTS.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer, TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import FunctionTransformer, Normalizer

def _tokens(t):
    return re.findall(r"\w+", str(t).lower())

def _meta(texts):
    s = pd.Series(list(texts)).fillna("").astype(str)
    n = s.str.len().clip(lower=1)
    return sparse.csr_matrix(pd.DataFrame({"len": np.log1p(n), "words": np.log1p(s.str.split().str.len()), "upper": s.str.count(r"[A-Z]") / n, "digit": s.str.count(r"\d") / n,
                                           "punct": s.str.count(r"[^\w\s]") / n, "excl": s.str.count("!").clip(upper=5), "url": s.str.count(r"https?://|URL"), "mention": s.str.count(r"@\w+|USER")}).to_numpy("float32"))

class RawText(BaseEstimator, TransformerMixin):
    """Teks mentah untuk model neural."""
    def fit(self, X, y=None): return self
    def transform(self, X): return list(X)

class GensimVec(BaseEstimator, TransformerMixin):
    """kind: w2v, fasttext, doc2vec. idf=True: rata-rata berbobot."""
    def __init__(self, kind="w2v", dim=200, window=5, min_count=2, epochs=5, idf=False, workers=4, seed=42):
        self.kind, self.dim, self.window, self.min_count, self.epochs, self.idf, self.workers, self.seed = kind, dim, window, min_count, epochs, idf, workers, seed
    def fit(self, X, y=None):
        from gensim.models import FastText, Word2Vec
        from gensim.models.doc2vec import Doc2Vec, TaggedDocument
        toks = [_tokens(t) for t in X]
        kw = dict(vector_size=self.dim, min_count=self.min_count, epochs=self.epochs, workers=self.workers, seed=self.seed)
        if self.kind == "doc2vec":
            self.m_ = Doc2Vec([TaggedDocument(t, [i]) for i, t in enumerate(toks)], dm=0, **kw)
        else:
            self.m_ = (FastText if self.kind == "fasttext" else Word2Vec)(toks, window=self.window, **kw).wv
        self.w_ = None
        if self.idf:
            t = TfidfVectorizer(token_pattern=r"\w+").fit(X)
            self.w_ = dict(zip(t.get_feature_names_out(), t.idf_))
        return self
    def transform(self, X):
        out = np.zeros((len(X), self.dim), dtype="float32")
        for i, t in enumerate(X):
            w = _tokens(t)
            if self.kind == "doc2vec":
                out[i] = self.m_.infer_vector(w)
                continue
            w = w if self.kind == "fasttext" else [x for x in w if x in self.m_]
            if w:
                a = np.array([self.w_.get(x, 1.0) for x in w])[:, None] if self.w_ else np.ones((len(w), 1))
                out[i] = (self.m_[w] * a).sum(0) / a.sum()
        return out

class EmbedVec(BaseEstimator, TransformerMixin):
    """Embedding SentenceTransformer dengan cache per teks agar tiap fold tidak encode ulang."""
    _cache = {}
    def __init__(self, model_path=None, prefix="", batch=128):
        self.model_path, self.prefix, self.batch = model_path, prefix, batch
    def fit(self, X, y=None): return self
    def transform(self, X):
        c = EmbedVec._cache.setdefault(str(self.model_path), {})
        miss = [t for t in dict.fromkeys(map(str, X)) if t not in c]
        if miss:
            from sentence_transformers import SentenceTransformer
            m = SentenceTransformer(str(self.model_path), local_files_only=True)
            for t, v in zip(miss, m.encode([self.prefix + t for t in miss], batch_size=self.batch, normalize_embeddings=True, show_progress_bar=False)):
                c[t] = v.astype("float32")
        return np.vstack([c[str(t)] for t in X])

def make_vectorizer(kind="tfidf_union", **kw):
    """Semua hasilnya punya fit_transform dan transform."""
    word = dict(ngram_range=(1, 2), min_df=2, max_df=.95, sublinear_tf=True, dtype="float32", token_pattern=r"(?u)\b\w+\b")
    char = dict(analyzer="char_wb", ngram_range=(2, 5), min_df=3, max_features=500_000, sublinear_tf=True, dtype="float32")
    if kind == "tfidf_word": return TfidfVectorizer(**{**word, **kw})
    if kind == "tfidf_char": return TfidfVectorizer(**{**char, **kw})
    if kind == "tfidf_union": return make_union(TfidfVectorizer(**word), TfidfVectorizer(**char))
    if kind == "count": return CountVectorizer(**{**dict(ngram_range=(1, 1), min_df=2, max_df=.95), **kw})
    if kind == "binary": return CountVectorizer(**{**dict(ngram_range=(1, 2), min_df=2, binary=True), **kw})
    if kind == "hashing": return HashingVectorizer(**{**dict(n_features=2 ** 20, alternate_sign=False, ngram_range=(1, 2), dtype="float32"), **kw})
    if kind == "lsa": return make_pipeline(TfidfVectorizer(**word), TruncatedSVD(kw.pop("k", 256), random_state=42), Normalizer(copy=False))
    if kind == "meta": return FunctionTransformer(_meta, validate=False)
    if kind == "tfidf_meta": return make_union(TfidfVectorizer(**word), FunctionTransformer(_meta, validate=False))
    if kind in ("w2v", "fasttext", "doc2vec"): return GensimVec(kind, **kw)
    if kind == "w2v_idf": return GensimVec("w2v", idf=True, **kw)
    if kind == "embed": return EmbedVec(**kw)
    if kind == "raw": return RawText()
    raise ValueError(f"vectorizer tidak dikenal: {kind}")

# ===== jalankan =====
print(make_vectorizer("tfidf_word").fit_transform(texts[:5000]).shape)


## Validasi

Target di-bin agar tiap fold mewakili rentang.

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, KFold, LeaveOneGroupOut, StratifiedGroupKFold, StratifiedKFold

def _ml_folds(Y, n, seed):
    raw = np.asarray(Y, dtype=float)
    assert np.isin(raw[np.isfinite(raw)], [0, 1]).all(), "Target harus 0/1/NaN"
    Y, rng = np.hstack([raw == 1, np.isfinite(raw)]), np.random.default_rng(seed)
    left, fold = np.ones(len(Y), bool), -np.ones(len(Y), int)
    need, cap = np.tile(Y.sum(0) / n, (n, 1)).astype(float), np.full(n, len(Y) / n)
    while left.any():
        c = Y[left].sum(0)
        c[c == 0] = 10 ** 9
        lab = int(c.argmin())
        pool = np.where(left)[0] if c[lab] == 10 ** 9 else rng.permutation(np.where(left & Y[:, lab])[0])
        for i in pool:
            d = need[:, lab]
            cand = np.where(d == d.max())[0]
            f = int(cand[cap[cand].argmax()])
            fold[i], left[i] = f, False
            cap[f] -= 1
            need[f, Y[i]] -= 1
    return [(np.where(fold != k)[0], np.where(fold == k)[0]) for k in range(n)]

def make_folds(y=None, groups=None, times=None, kind="auto", n=5, seed=42, valid_share=.2):
    """Kembalikan list (idx_train, idx_valid). time: satu holdout terbaru."""
    if kind == "auto":
        y_ = None if y is None else np.asarray(y)
        if times is not None: kind = "time"
        elif y_ is None: kind = "group" if groups is not None else "kfold"
        elif y_.ndim == 2: kind = "multilabel"
        elif np.issubdtype(y_.dtype, np.floating) and len(np.unique(y_)) > 20: kind = "regression" if groups is None else "group"
        else: kind = "strat_group" if groups is not None else "stratified"
        print("skema fold:", kind)
    if kind == "time":
        t = pd.to_datetime(pd.Series(list(times)), errors="coerce")
        cut = t.quantile(1 - valid_share)
        return [(np.where(t < cut)[0], np.where(t >= cut)[0])]
    if kind == "source": return list(LeaveOneGroupOut().split(np.zeros(len(groups)), groups=groups))
    N = len(y) if y is not None else len(groups)
    Z = np.zeros(N)
    if kind == "multilabel":
        if groups is None: return _ml_folds(y, n, seed)
        gid, inverse = np.unique(groups, return_inverse=True)
        raw = np.asarray(y, float)
        profile = np.array([np.r_[np.any(raw[inverse==i]==1,axis=0), np.any(np.isfinite(raw[inverse==i]),axis=0)] for i in range(len(gid))])
        return [(np.flatnonzero(np.isin(inverse,a)),np.flatnonzero(np.isin(inverse,b))) for a,b in _ml_folds(profile,n,seed)]
    if kind == "kfold": return list(KFold(n, shuffle=True, random_state=seed).split(Z))
    if kind == "group": return list(GroupKFold(n, shuffle=True, random_state=seed).split(Z, groups=groups))
    if kind == "strat_group": return list(StratifiedGroupKFold(n, shuffle=True, random_state=seed).split(Z, y, groups))
    if kind == "regression":
        v = np.asarray(y, float)
        return list(StratifiedKFold(n, shuffle=True, random_state=seed).split(Z, np.digitize(v, np.unique(np.quantile(v, np.linspace(0, 1, 11)[1:-1])))))
    if kind == "stratified": return list(StratifiedKFold(n, shuffle=True, random_state=seed).split(Z, y))
    raise ValueError(kind)

def check_folds(texts, folds):
    """Persen teks validasi yang juga ada di latih. Harus mendekati 0 kecuali test memang overlap."""
    k = np.array([re.sub(r"\W+", " ", str(t).lower()).strip() for t in texts], dtype=object)
    r = [round(float(np.isin(k[b], k[a]).mean()), 4) for a, b in folds]
    print("overlap per fold:", r)
    return r

def _proba(m, X):
    if hasattr(m, "predict_proba"):
        return m.predict_proba(X)
    if hasattr(m, "decision_function"):
        raise ValueError('Margin bukan probabilitas. Gunakan predict untuk F1 atau classifier terkalibrasi untuk probability ensemble.')
    return np.asarray(m.predict(X)).reshape(-1, 1)

def oof_predict(make_vec, make_mdl, texts, y, folds, test_texts=None, sample_weight=None):
    """make_vec, make_mdl: fungsi tanpa argumen. Vectorizer di-fit di dalam fold. Kembalikan (oof, test rata-rata fold)."""
    texts, y = np.asarray(texts, dtype=object), np.asarray(y)
    oof, test = None, None
    for tr, va in folds:
        vec, model = make_vec(), make_mdl()
        Xtr, Xva = vec.fit_transform(list(texts[tr]), y[tr]), vec.transform(list(texts[va]))
        kw = {} if sample_weight is None else {"sample_weight": np.asarray(sample_weight)[tr]}
        model.fit(Xtr, y[tr], **kw)
        p = _proba(model, Xva)
        classes = np.unique(y) if y.ndim == 1 and hasattr(model, 'classes_') else None
        if classes is not None:
            aligned = np.zeros((len(va),len(classes)))
            aligned[:,np.searchsorted(classes,model.classes_)] = p
            p = aligned
        if oof is None:
            oof = np.full((len(y), p.shape[1]), np.nan, dtype="float32")
        oof[va] = p
        if test_texts is not None:
            t = _proba(model, vec.transform(list(test_texts)))
            if classes is not None:
                aligned = np.zeros((len(t),len(classes)))
                aligned[:,np.searchsorted(classes,model.classes_)] = t
                t = aligned
            test = t / len(folds) if test is None else test + t / len(folds)
    return oof, test

# ===== jalankan =====
y = train[TARGET].to_numpy(float)
folds = make_folds(y, kind="regression", seed=SEED)
check_folds(texts, folds)


## Model

Bandingkan dengan prediksi rata-rata.

In [ ]:
import re
from collections import Counter
from functools import partial
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin

def _net(kind, vocab, n_out, dim=200, hidden=128, drop=.3):
    import torch
    import torch.nn as nn
    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.emb, self.drop = nn.Embedding(vocab, dim, padding_idx=0), nn.Dropout(drop)
            if kind == "cnn":
                self.conv = nn.ModuleList(nn.Conv1d(dim, hidden, k, padding=k // 2) for k in (3, 4, 5))
                self.out = nn.Linear(hidden * 3, n_out)
            else:
                bi = kind in ("bigru", "bilstm", "attn")
                self.rnn = (nn.LSTM if "lstm" in kind else nn.GRU)(dim, hidden, batch_first=True, bidirectional=bi)
                self.att = nn.Linear(hidden * (2 if bi else 1), 1)
                self.out = nn.Linear(hidden * (2 if bi else 1), n_out)
        def forward(self, x):
            e = self.emb(x)
            if kind == "cnn":
                return self.out(self.drop(torch.cat([c(e.transpose(1, 2)).relu().amax(2) for c in self.conv], 1)))
            h, _ = self.rnn(e)
            w = self.att(h).squeeze(-1).masked_fill(x == 0, -1e4).softmax(1) if kind == "attn" else (x != 0).float() / (x != 0).sum(1, keepdim=True).clamp(min=1)
            return self.out(self.drop((h * w.unsqueeze(-1)).sum(1)))
    return Net()

class NeuralText(BaseEstimator, ClassifierMixin):
    """Terima list teks. Pakai bersama make_vectorizer('raw')."""
    def __init__(self, kind="cnn", epochs=3, batch=128, lr=1e-3, max_len=128, min_count=2, device=None, seed=42):
        self.kind, self.epochs, self.batch, self.lr, self.max_len, self.min_count, self.device, self.seed = kind, epochs, batch, lr, max_len, min_count, device, seed
    def _ids(self, T):
        a = np.zeros((len(T), self.max_len), dtype="int64")
        for i, t in enumerate(T):
            w = [self.v_.get(x, 1) for x in re.findall(r"\w+", str(t).lower())[:self.max_len]]
            a[i, :len(w)] = w
        return a
    def fit(self, T, y):
        import torch
        torch.manual_seed(self.seed)
        self.d_ = self.device or ("cuda" if torch.cuda.is_available() else "cpu")
        c = Counter(w for t in T for w in re.findall(r"\w+", str(t).lower()))
        self.v_ = {w: i + 2 for i, (w, k) in enumerate(c.most_common()) if k >= self.min_count}
        self.classes_, yy = np.unique(y, return_inverse=True)
        X, yy = torch.tensor(self._ids(T)), torch.tensor(yy)
        self.n_ = _net(self.kind, len(self.v_) + 2, len(self.classes_)).to(self.d_)
        opt, loss = torch.optim.AdamW(self.n_.parameters(), lr=self.lr), torch.nn.CrossEntropyLoss()
        for _ in range(self.epochs):
            self.n_.train()
            for b in torch.randperm(len(yy)).split(self.batch):
                opt.zero_grad(); loss(self.n_(X[b].to(self.d_)), yy[b].to(self.d_)).backward(); opt.step()
        return self
    def predict_proba(self, T):
        import torch
        self.n_.eval()
        with torch.inference_mode():
            return np.vstack([self.n_(b.to(self.d_)).softmax(1).float().cpu().numpy() for b in torch.tensor(self._ids(T)).split(512)])
    def predict(self, T):
        return self.classes_[self.predict_proba(T).argmax(1)]

def make_model(name="lr", seed=42, n_jobs=8, **kw):
    if name in ("cnn", "gru", "bigru", "lstm", "bilstm", "attn"):
        return NeuralText(name, seed=seed, **kw)
    from sklearn.linear_model import LogisticRegression, Ridge, RidgeClassifier, SGDClassifier
    if name == "lr": return LogisticRegression(**{**dict(C=4.0, max_iter=2000), **kw})
    if name == "sgd": return SGDClassifier(**{**dict(loss="log_loss", alpha=1e-6, random_state=seed), **kw})
    if name == "ridge_clf": return RidgeClassifier(**kw)
    if name in ("svc", "svc_cal"):
        from sklearn.svm import LinearSVC
        m = LinearSVC(**{**dict(C=.5, dual="auto", max_iter=5000), **kw})
        if name == "svc":
            return m
        from sklearn.calibration import CalibratedClassifierCV
        return CalibratedClassifierCV(m, method="sigmoid", cv=3)
    if name in ("cnb", "mnb"):
        from sklearn.naive_bayes import ComplementNB, MultinomialNB
        return (ComplementNB if name == "cnb" else MultinomialNB)(**{**dict(alpha=.3), **kw})
    if name in ("lgbm", "lgbm_reg", "lgbm_l1"):
        import lightgbm as lgb
        p = {**dict(n_estimators=500, learning_rate=.05, num_leaves=63, subsample=.8, subsample_freq=1, colsample_bytree=.5, random_state=seed, n_jobs=n_jobs, verbose=-1), **kw}
        return lgb.LGBMClassifier(**p) if name == "lgbm" else lgb.LGBMRegressor(objective="l1" if name == "lgbm_l1" else "l2", **p)
    if name == "xgb":
        from xgboost import XGBClassifier
        return XGBClassifier(**{**dict(n_estimators=500, learning_rate=.05, max_depth=6, tree_method="hist", subsample=.8, colsample_bytree=.5, random_state=seed), **kw})
    if name in ("rf", "rf_reg"):
        from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
        return (RandomForestClassifier if name == "rf" else RandomForestRegressor)(**{**dict(n_estimators=300, min_samples_leaf=2, n_jobs=n_jobs, random_state=seed), **kw})
    if name == "mlp":
        from sklearn.neural_network import MLPClassifier
        return MLPClassifier(**{**dict(hidden_layer_sizes=(256,), early_stopping=True, max_iter=100, random_state=seed), **kw})
    if name == "smote_lr":
        from imblearn.over_sampling import SMOTE
        from imblearn.pipeline import make_pipeline as imb
        return imb(SMOTE(k_neighbors=kw.get("k", 5), random_state=seed), LogisticRegression(max_iter=2000))
    if name in ("ovr", "chain"):
        base = LogisticRegression(**{**dict(C=4.0, max_iter=2000), **kw})
        if name == "ovr":
            from sklearn.multiclass import OneVsRestClassifier
            return OneVsRestClassifier(base, n_jobs=4)
        from sklearn.multioutput import ClassifierChain
        return ClassifierChain(base, order="random", random_state=seed)
    if name in ("ridge", "ridge_log"):
        m = Ridge(**{**dict(alpha=1.0, solver="sag", random_state=seed), **kw})
        if name == "ridge":
            return m
        from sklearn.compose import TransformedTargetRegressor
        return TransformedTargetRegressor(m, func=np.log1p, inverse_func=np.expm1)
    if name in ("dummy", "dummy_reg"):
        from sklearn.dummy import DummyClassifier, DummyRegressor
        return DummyClassifier(strategy="prior") if name == "dummy" else DummyRegressor(strategy=kw.get("strategy", "mean"))
    raise ValueError(f"model tidak dikenal: {name}")

import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import cohen_kappa_score, mean_absolute_error, r2_score, root_mean_squared_error

def reg_report(y, pred):
    y, pred = np.asarray(y, float), np.asarray(pred, float)
    r = dict(rmse=root_mean_squared_error(y, pred), mae=mean_absolute_error(y, pred), r2=r2_score(y, pred), pearson=pearsonr(y, pred)[0], spearman=spearmanr(y, pred)[0])
    if (y > 0).all():
        r["mape"] = float(np.mean(np.abs((y - pred) / y)))
    if (y >= 0).all() and (pred >= 0).all():
        r["rmsle"] = root_mean_squared_error(np.log1p(y), np.log1p(pred))
    return {k: round(float(v), 4) for k, v in r.items()}

def qwk(y, pred):
    return float(cohen_kappa_score(np.asarray(y).astype(int), np.asarray(pred).astype(int), weights="quadratic"))

import json
import time
from pathlib import Path
import numpy as np
import pandas as pd

def save_run(name, oof=None, test=None, score=None, config=None, ids=None, folder="runs"):
    d = Path(folder) / name
    d.mkdir(parents=True, exist_ok=True)
    if oof is not None:
        np.save(d / "oof.npy", oof)
    if test is not None:
        np.save(d / "test.npy", test)
    if ids is not None:
        np.save(d / "ids.npy", np.asarray(ids))
    (d / "meta.json").write_text(json.dumps({"score": score, "config": config, "time": time.strftime("%H:%M:%S")}, default=str, indent=1))
    with open(Path(folder) / "log.csv", "a", encoding="utf-8") as f:
        f.write(f"{time.strftime('%H:%M:%S')},{name},{score}\n")

def load_run(name, folder="runs"):
    d = Path(folder) / name
    return {p.stem: np.load(p, allow_pickle=True) for p in d.glob("*.npy")} | json.loads((d / "meta.json").read_text())

def leaderboard(folder="runs"):
    return pd.read_csv(Path(folder) / "log.csv", names=["waktu", "run", "skor"]).sort_values("skor", ascending=False)

# ===== jalankan =====
results = {}
print("baseline", reg_report(y, np.r_[np.full(len(y) - 1, y.mean()), y.mean() + 1e-6]))
for v, m in EXPERIMENTS:
    o, p = oof_predict(lambda: make_vectorizer(v, **(dict(k=64) if v == "lsa" else {})), lambda: make_model(m), texts, y, folds, test_texts)
    results[f"{v}+{m}"] = (o.ravel(), p.ravel())
    print(f"{v}+{m}", reg_report(y, o.ravel()))
    save_run(f"{v}+{m}", o, p, reg_report(y, o.ravel())["rmse"])
best = min(results, key=lambda k: reg_report(y, results[k][0])["rmse"])
oof, pred = results[best]
print("terbaik", best)


## Transformer

Lewati bila USE_BERT=False.

In [ ]:
from pathlib import Path
import numpy as np
import torch
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments)

class TextDS(torch.utils.data.Dataset):
    def __init__(self, enc, labels=None):
        self.enc, self.labels = enc, labels
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, i):
        d = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None:
            d["labels"] = self.labels[i]
        return d

class LossTrainer(Trainer):
    """loss_fn(logits, labels) opsional, misalnya focal loss."""
    def __init__(self, *a, loss_fn=None, **k):
        super().__init__(*a, **k)
        self.loss_fn = loss_fn
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        if self.loss_fn is None:
            return super().compute_loss(model, inputs, return_outputs=return_outputs, **kw)
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = self.loss_fn(out.logits, labels)
        return (loss, out) if return_outputs else loss

def train_transformer(model_path, texts, y, valid_texts=None, valid_y=None, texts2=None, valid_texts2=None, problem="single",
                      num_labels=None, max_len=128, batch=16, accum=1, epochs=1, lr=2e-5, warmup=.1, eval_every=500, patience=3,
                      out="run_tf", seed=42, loss_fn=None, smoothing=0.0, resume=False, fp16=False):
    tok = AutoTokenizer.from_pretrained(str(model_path), local_files_only=True)
    def enc(a, b):
        return tok([str(t) for t in a], None if b is None else [str(t) for t in b], truncation=True, max_length=max_len)
    def lab(v):
        v = np.asarray(v)
        return v.astype("float32").tolist() if problem != "single" else v.astype("int64").tolist()
    n = num_labels or (1 if problem == "reg" else np.asarray(y).shape[1] if problem == "multi" else int(np.max(y)) + 1)
    ptype = {"single": "single_label_classification", "multi": "multi_label_classification", "reg": "regression"}[problem]
    model = AutoModelForSequenceClassification.from_pretrained(str(model_path), local_files_only=True, num_labels=n, problem_type=ptype, ignore_mismatched_sizes=True)
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported() and not fp16
    has_val = valid_texts is not None
    args = TrainingArguments(output_dir=out, per_device_train_batch_size=batch, per_device_eval_batch_size=batch * 2, gradient_accumulation_steps=accum,
                             num_train_epochs=epochs, learning_rate=lr, warmup_steps=warmup, label_smoothing_factor=smoothing,
                             eval_strategy="steps" if has_val else "no", eval_steps=eval_every, save_strategy="steps" if has_val else "no", save_steps=eval_every,
                             save_total_limit=2, load_best_model_at_end=has_val, metric_for_best_model="eval_loss", greater_is_better=False,
                             train_sampling_strategy="group_by_length", bf16=use_bf16, fp16=fp16 and torch.cuda.is_available(),
                             dataloader_num_workers=0, report_to="none", logging_steps=100, seed=seed, disable_tqdm=True)
    tr = LossTrainer(model=model, args=args, train_dataset=TextDS(enc(texts, texts2), lab(y)),
                     eval_dataset=TextDS(enc(valid_texts, valid_texts2), lab(valid_y)) if has_val else None,
                     processing_class=tok, data_collator=DataCollatorWithPadding(tok), loss_fn=loss_fn,
                     callbacks=[EarlyStoppingCallback(patience)] if has_val else None)
    if loss_fn is not None:
        tr.model_accepts_loss_kwargs = False
    from transformers.trainer_utils import get_last_checkpoint
    last = get_last_checkpoint(out) if resume and Path(out).is_dir() else None
    if last:
        print("lanjut dari", last)
    tr.train(resume_from_checkpoint=last)
    return tr.model, tok

@torch.inference_mode()
def predict_transformer(model, tok, texts, texts2=None, problem="single", max_len=128, batch=64):
    """Terurut panjang untuk hemat padding. single: softmax, multi: sigmoid, reg: nilai mentah."""
    dev = next(model.parameters()).device
    model.eval()
    texts = [str(t) for t in texts]
    order = np.argsort([len(t) for t in texts])
    out = None
    for i in range(0, len(texts), batch):
        idx = order[i:i + batch]
        a, b = [texts[j] for j in idx], None if texts2 is None else [str(texts2[j]) for j in idx]
        e = tok(a, b, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(dev)
        lg = model(**e).logits
        p = lg.softmax(1) if problem == "single" else lg.sigmoid() if problem == "multi" else lg
        p = p.float().cpu().numpy()
        if out is None:
            out = np.zeros((len(texts), p.shape[1]), dtype="float32")
        out[idx] = p
    return out

def train_regression(model_path, texts, y, valid_texts=None, valid_y=None, texts2=None, valid_texts2=None, **kw):
    y = np.asarray(y, dtype="float32")
    mu, sd = y.mean(), y.std() + 1e-6
    vy = None if valid_y is None else (np.asarray(valid_y, dtype="float32") - mu) / sd
    m, t = train_transformer(model_path, texts, ((y - mu) / sd)[:, None], valid_texts, None if vy is None else vy[:, None], texts2, valid_texts2, problem="reg", num_labels=1, **kw)
    return m, t, (float(mu), float(sd))

def predict_regression(model, tok, texts, scale, texts2=None, **kw):
    return predict_transformer(model, tok, texts, texts2, problem="reg", **kw).ravel() * scale[1] + scale[0]

# ===== jalankan =====
bert_test = None
if USE_BERT:
    tr, va = folds[0]
    m, tok, sc = train_regression(BERT_MODEL, [texts[i] for i in tr], y[tr], [texts[i] for i in va], y[va], max_len=MAX_LEN, batch=BATCH)
    print("BERT", reg_report(y[va], predict_regression(m, tok, [texts[i] for i in va], sc, max_len=MAX_LEN)))
    bert_test = predict_regression(m, tok, test_texts, sc, max_len=MAX_LEN)
else:
    print("USE_BERT=False")


## Evaluasi & error analysis

Residual per rentang target dan noise floor.

In [ ]:
import numpy as np

def _sample(n, groups, rng):
    if groups is None:
        return rng.integers(0, n, n)
    g = np.asarray(groups)
    u = np.unique(g)
    pick = rng.choice(u, len(u))
    pos = {k: np.where(g == k)[0] for k in u}
    return np.concatenate([pos[k] for k in pick])

def noise_floor(y, pred, score_fn, n_boot=300, groups=None, seed=42):
    """Kembalikan (skor, SE, batas 2,5%, batas 97,5%). groups untuk cluster bootstrap."""
    y, pred, rng = np.asarray(y), np.asarray(pred), np.random.default_rng(seed)
    s = [score_fn(y[i], pred[i]) for i in (_sample(len(y), groups, rng) for _ in range(n_boot))]
    return round(score_fn(y, pred), 4), round(float(np.std(s)), 4), round(float(np.percentile(s, 2.5)), 4), round(float(np.percentile(s, 97.5)), 4)

def paired_gain(y, pred_a, pred_b, score_fn, n_boot=300, groups=None, seed=42):
    """B dibanding A pada baris yang sama. Terima B bila gain > 2*se."""
    y, a, b, rng = np.asarray(y), np.asarray(pred_a), np.asarray(pred_b), np.random.default_rng(seed)
    d = []
    for _ in range(n_boot):
        i = _sample(len(y), groups, rng)
        d.append(score_fn(y[i], b[i]) - score_fn(y[i], a[i]))
    return dict(gain=round(score_fn(y, b) - score_fn(y, a), 4), se=round(float(np.std(d)), 4), p_b_lebih_baik=round(float(np.mean(np.array(d) > 0)), 3))

def learning_curve_oof(fit_score, n_rows, shares=(.1, .25, .5, 1.0), seed=42):
    """fit_score(idx) melatih pada idx dan mengembalikan skor validasi tetap."""
    order = np.random.default_rng(seed).permutation(n_rows)
    return {s: fit_score(order[:int(n_rows * s)]) for s in shares}

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score

def worst_errors(texts, y, proba, k=30, names=None):
    y, p = np.asarray(y), np.asarray(proba)
    pred = p.argmax(1)
    d = pd.DataFrame({"text": list(texts), "y": y, "pred": pred, "conf": p.max(1)})[y != pred]
    if names is not None:
        d["y"], d["pred"] = d.y.map(dict(enumerate(names))), d.pred.map(dict(enumerate(names)))
    return d.sort_values("conf", ascending=False).head(k)

def random_errors(texts, y, pred, k=50, seed=42):
    d = pd.DataFrame({"text": list(texts), "y": np.asarray(y), "pred": np.asarray(pred)})
    d = d[d.y != d.pred]
    return d.sample(min(k, len(d)), random_state=seed).assign(kategori="")

def confused_pairs(y, pred, names=None, k=10):
    c = confusion_matrix(y, pred, normalize="true")
    np.fill_diagonal(c, 0)
    o = np.dstack(np.unravel_index(np.argsort(-c.ravel()), c.shape))[0][:k]
    return [((names[a] if names is not None else int(a)), (names[b] if names is not None else int(b)), round(float(c[a, b]), 3)) for a, b in o]

def per_label(y, pred, names=None):
    return pd.DataFrame(classification_report(y, pred, target_names=names, zero_division=0, output_dict=True)).T.round(3)

def segment_scores(y, pred, segments, score=lambda a, b: f1_score(a, b, average="macro", zero_division=0), min_n=30):
    d = pd.DataFrame({"y": np.asarray(y), "p": np.asarray(pred), "s": np.asarray(segments)})
    r = {k: (len(g), round(score(g.y, g.p), 4)) for k, g in d.groupby("s") if len(g) >= min_n}
    return pd.DataFrame(r, index=["n", "skor"]).T.sort_values("skor")

def residual_table(y, pred, bins=5):
    d = pd.DataFrame({"y": np.asarray(y, float), "p": np.asarray(pred, float)})
    d["err"] = d.p - d.y
    return d.groupby(pd.qcut(d.y, bins, duplicates="drop"), observed=True).err.agg(["count", "mean", "std"]).round(3)

# ===== jalankan =====
from sklearn.metrics import root_mean_squared_error
print(noise_floor(y, oof, root_mean_squared_error, 200))
print(residual_table(y, oof).to_string())


## Post-processing

Blend, cut point ordinal, clip.

In [ ]:
import numpy as np
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import KFold

def apply_cuts(p, cuts, low=0):
    return low + np.digitize(np.asarray(p), np.sort(cuts))

def fit_cuts(y, p, low=None, rounds=2, n_grid=31):
    y, p = np.asarray(y).astype(int), np.asarray(p, float)
    low = int(y.min()) if low is None else low
    k = int(y.max()) - low
    cuts = np.quantile(p, np.cumsum(np.bincount(y - low))[:-1] / len(y))
    score = lambda c: cohen_kappa_score(y, apply_cuts(p, c, low), weights="quadratic")
    best = score(cuts)
    for _ in range(rounds):
        for i in range(k):
            for g in np.linspace(cuts[i] - .5 * p.std(), cuts[i] + .5 * p.std(), n_grid):
                c = cuts.copy(); c[i] = g
                s = score(c)
                if s > best + 1e-5:
                    best, cuts = s, c
    return np.sort(cuts)

def nested_cuts_score(y, p, n=5, seed=42):
    y, p = np.asarray(y).astype(int), np.asarray(p, float)
    pred = np.zeros(len(y), int)
    low = int(y.min())
    for a, b in KFold(n, shuffle=True, random_state=seed).split(p):
        pred[b] = apply_cuts(p[b], fit_cuts(y[a], p[a], low=low), low)
    return round(cohen_kappa_score(y, pred, weights="quadratic"), 4)

def to_output(p, log=False, lo=None, hi=None, integer=False):
    p = np.expm1(p) if log else np.asarray(p)
    p = np.clip(p, lo, hi) if lo is not None or hi is not None else p
    return np.rint(p).astype(int) if integer else p

import numpy as np

def blend(preds, weights=None):
    """preds: list array sama bentuk. Bekerja untuk probabilitas, nilai regresi, dan bagging seed."""
    return np.average(np.stack(preds), axis=0, weights=weights)

# ===== jalankan =====
if bert_test is not None:
    pred = blend([pred, bert_test])
if ORDINAL:
    print("QWK bulat", round(qwk(y, np.rint(oof)), 4), "| cut nested", nested_cuts_score(y, oof))
    final = apply_cuts(pred, fit_cuts(y, oof), low=int(y.min()))
else:
    final = to_output(pred, lo=y.min(), hi=y.max())


## Submission

Kolom terakhir contoh submission.

In [ ]:
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd

def encode_labels(y):
    """Kembalikan (y_int, kelas). Decode: kelas[pred]."""
    classes, yi = np.unique(np.asarray(y).astype(str), return_inverse=True)
    return yi, classes

def write_submission(sample_path, preds, out="submission.csv", pred_col=None, id_col=None, allowed=None):
    """Salin struktur contoh submission, isi prediksi, validasi, dan cetak hash."""
    s = pd.read_csv(sample_path, dtype=str)
    id_col = id_col or s.columns[0]
    pred_col = pred_col or s.columns[-1]
    assert len(s) == len(preds), f"jumlah baris {len(preds)} != contoh {len(s)}"
    s[pred_col] = preds
    assert s[pred_col].notna().all(), "ada NaN"
    assert s[id_col].is_unique, "id ganda"
    if allowed is not None:
        bad = set(s[pred_col]) - set(allowed)
        assert not bad, f"nilai asing: {bad}"
    s.to_csv(out, index=False)
    print(out, s.shape, "sha256", hashlib.sha256(Path(out).read_bytes()).hexdigest()[:16], s[pred_col].value_counts(normalize=True).head(5).round(3).to_dict())
    return s

# ===== jalankan =====
sub = write_submission(SAMPLE, final)
